<a href="https://colab.research.google.com/github/claudio1975/Medium-blog/blob/master/Fine-tuning_LLMS_on_medical_dataset_with_Unsloth/Qwen3_4B_Instruct_fine_tuned_OpenMed_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# STEP 1: INSTALL DEPENDENCIES
# ============================================================================


print("📦 Installing Unsloth and dependencies...")
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes
print("✅ Installation complete!")


In [ ]:
# ============================================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================================

print("📚 Importing libraries...")
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments, TextStreamer
import pandas as pd
import os
import json

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


📚 Importing libraries...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ PyTorch version: 2.9.1+cu128
✅ CUDA available: True
✅ GPU: NVIDIA A100-SXM4-40GB
✅ GPU Memory: 39.56 GB


In [ ]:
# ============================================================================
# STEP 3: LOAD MODEL
# ============================================================================

print("\n⚙️ Setting up configuration...")

# Model settings
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3-4B-Instruct-2507",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = False,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    dtype = torch.float16,
    full_finetuning = False,

)



In [ ]:
# ============================================================================
# STEP 4: LORA ADAPTERS
# ============================================================================


model = FastLanguageModel.get_peft_model(
    model,
    r = 32,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"✅ Total parameters: {total_params:,}")


Unsloth 2025.12.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


✅ Trainable parameters: 66,060,288 (1.62%)
✅ Total parameters: 4,088,528,384


In [ ]:
# ============================================================================
# STEP 5: LOAD AND INSPECT DATASET
# ============================================================================

print("\n📊 Loading dataset: OpenMed/Medical-Reasoning-SFT-GPT-OSS-120B")
print("⏳ This may take a moment...")

# Load dataset
dataset = load_dataset("OpenMed/Medical-Reasoning-SFT-GPT-OSS-120B", split="train")

print(f"✅ Dataset loaded! Total samples: {len(dataset):,}")
print(f"\n📋 Dataset columns: {dataset.column_names}")
print(f"\n🔍 First sample:")
print(dataset[0])


In [ ]:
# ============================================================================
# STEP 6: FORMAT DATASET
# ============================================================================

def format_conversation(example):
    conversation = ""
    for msg in example["messages"]:
        role = msg["role"]
        content = msg["content"]
        conversation += f"{role}: {content}\n"
    return {"text": conversation}

formatted_dataset = dataset.map(format_conversation)

In [ ]:
formatted_dataset[0]

In [ ]:
data = pd.Series(formatted_dataset)
data.name = "text"


In [ ]:
# ============================================================================
# STEP 7: CONFIGURE TRAINER
# ============================================================================

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 8, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 100,
        save_steps=500,
        optim = "adamw_torch",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
# ============================================================================
# STEP 8: TRAIN!
# ============================================================================

print("\n" + "="*70)
print("🚀 STARTING TRAINING!")
print("="*70)
print("="*70 + "\n")

# Show GPU memory before training
if torch.cuda.is_available():
    print(f"💾 GPU Memory before training: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# Train!
trainer_stats = trainer.train()

print("\n" + "="*70)
print("🎉 TRAINING COMPLETED!")
print("="*70)
print(f"⏱️  Total training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"⚡ Samples/second: {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"📉 Final loss: {trainer_stats.metrics['train_loss']:.4f}")
print("="*70 + "\n")



In [ ]:
messages = [
    {"role" : "user", "content" : "What are the main symptoms of pneumonia?"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 256, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

The main symptoms of pneumonia can vary depending on the age of the person and the type of pneumonia, but common symptoms include:

1. **Cough** – Often productive (with phlegm), which may be yellow, green, or bloody.
2. **Fever** – Usually high, though it can be low-grade or absent in older adults or young children.
3. **Shortness of breath** – Difficulty breathing, especially during physical activity or when lying down.
4. **Chest pain** – A sharp or stabbing pain that worsens when coughing or breathing deeply.
5. **Fatigue** – Feeling unusually tired or weak.
6. **Loss of appetite** – Reduced interest in food.
7. **Confusion** – Especially in older adults, confusion can be an early sign.
8. **Chills and sweating** – Often accompanying fever.
9. **Nausea or vomiting** – More common in children or those with severe cases.

In infants and young children, symptoms may include irritability, rapid breathing, poor feeding, and a bluish tint to the skin (cyanosis).

If you or someone you kn

In [ ]:
messages = [
    {"role" : "user", "content" : "What are the main symptoms of diabetes"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 256, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Diabetes is a chronic condition that affects how the body processes blood sugar (glucose). The main symptoms of diabetes—especially type 1 and type 2—can vary, but common signs include:

1. **Frequent Urination (Polyuria)**  
   The body tries to eliminate excess glucose through urine, leading to increased urination, especially at night (nocturia).

2. **Excessive Thirst (Polydipsia)**  
   As the body loses fluids through frequent urination, you may feel unusually thirsty and crave water.

3. **Increased Hunger (Polyphagia)**  
   Even after eating, the body may not get enough energy due to poor glucose utilization, leading to persistent hunger.

4. **Unexplained Weight Loss**  
   In type 1 diabetes, the body starts breaking down fat and muscle for energy, leading to rapid weight loss despite eating normally.

5. **Fatigue**  
   Without proper glucose uptake by cells, tissues don’t get the energy they need, resulting in persistent tiredness.

6. **Blurred Vision**  
   High blood su

In [ ]:
messages = [
    {"role" : "user", "content" : "What are the main symptoms of heart disease?"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 256, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Heart disease encompasses a range of conditions affecting the heart, and symptoms can vary depending on the specific type. However, some of the most common symptoms of heart disease include:

1. **Chest Pain or Discomfort (Angina)**  
   - A feeling of pressure, squeezing, fullness, or pain in the center or left side of the chest.  
   - May radiate to the arms, back, neck, jaw, or stomach.  
   - Often triggered by physical activity or stress and relieved by rest or medication.

2. **Shortness of Breath (Dyspnea)**  
   - Difficulty breathing during physical activity or when lying flat (orthopnea).  
   - May occur even at rest in more severe cases.

3. **Fatigue**  
   - Unexplained tiredness, especially during daily activities.  
   - Can be a subtle symptom, especially in women or older adults.

4. **Heart Palpitations**  
   - A sensation of a rapid, fluttering, or irregular heartbeat.  
   - May be associated with arrhythmias or other heart rhythm disorders.

5. **Swelling (Edema

In [ ]:
# ============================================================================
# STEP 10: SAVE MODEL
# ============================================================================

print("💾 Saving model locally...")

# Save LoRA adapters
OUTPUT_DIR="qwen3-medical-output"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ Model saved to: {OUTPUT_DIR}")



💾 Saving model locally...
✅ Model saved to: qwen3-medical-output


In [ ]:

print("💾 Saving model on HF_HUB...")

model.push_to_hub("towardsinnovationlab/Qwen3-4B_Instruct-medical", token = "") # Online saving
tokenizer.push_to_hub("towardsinnovationlab/Qwen3-4B_Instruct-medical", token = "") # Online saving

print(f"✅ Model saved to: towardsinnovationlab/Qwen3-4B_Instruct-medical")
